### Imports

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.config import CONFIG
from src.models.nli_dataset import SLMNLIDatasetInference
from src.models.slm_classifier import load_finetuned_for_inference, predict_batch

### Constants

In [ ]:
TEST_DATA_PATH = Path('../data/test.csv')
OUTPUT_PATH    = Path('../output')
OUTPUT_FILE_NAME = 'Group_40_C.csv'

ADAPTER_PATH = Path('../models/slm_adapter')

### Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Run demo

In [ ]:
def run_demo() -> None:
    cfg_slm = CONFIG.slm
    cfg_ft  = CONFIG.slm_finetune

    print(f'Loading base model: {cfg_slm.model}')
    print(f'Loading adapter from {ADAPTER_PATH}...')
    model, tokeniser = load_finetuned_for_inference(ADAPTER_PATH, cfg_slm, cfg_ft)

    df = pd.read_csv(TEST_DATA_PATH)
    # SLMNLIDatasetInference expects a label column — use a dummy value for test data
    df['label'] = 0
    dataset = SLMNLIDatasetInference(df, tokeniser, cfg_slm.max_length)
    test_dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

    model.eval()
    all_preds = []

    print(f'Running SLM demo on {TEST_DATA_PATH}...')
    for batch in tqdm(test_dataloader, desc='Evaluating', unit='batches'):
        preds = predict_batch(model, tokeniser, batch['input_ids'], batch['attention_mask'], device)
        all_preds.extend(preds)

    # Verify no results are missing
    assert len(all_preds) == len(dataset)
    result_df = pd.DataFrame({'prediction': all_preds})

    # Write to a csv file
    print(f'Done. Writing results to {OUTPUT_PATH / OUTPUT_FILE_NAME}.')
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    result_df.to_csv(OUTPUT_PATH / OUTPUT_FILE_NAME, index=False)

run_demo()